# 08 LSTM

Bidirectional PyTorch LSTM over historical ERCOT feature windows. Regression is required; the classifier uses the same backbone with a binary head.

In [1]:
# Cell 0 — Ensure PyTorch is available
!pip install torch --quiet

In [3]:
# Cell 1 — Shared scaffolding: imports, paths, load feature matrix, split, feature list
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

import copy
import gc

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from models.utils import (
    chronological_split,
    select_enhanced_features,
    TARGET_REG,
    TARGET_CLASS,
    regression_report,
    classification_report,
    apply_feature_transforms,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

FEATURES = PROJECT_ROOT / "data" / "features"

mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)
features = select_enhanced_features(mat)

print(f"\n{len(features)} features selected")
print("First 10:", features[:10], "...")
print("Last 10: ", features[-10:])

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)

63 features selected
First 10: ['da_HB_HOUSTON', 'da_HB_HUBAVG', 'da_HB_NORTH', 'da_HB_SOUTH', 'da_HB_WEST', 'day_of_week', 'gdelt_article_volume', 'gdelt_norm', 'gdelt_tone', 'gdelt_tone_change_1d'] ...
Last 10:  ['stress_gas_spike', 'stress_heat_wave', 'stress_low_wind', 'stress_reactor_outage', 'stress_score', 'temp_max_across_zones', 'temp_min_across_zones', 'temp_range_across_zones', 'wind_mean_across_zones', 'wind_min_across_zones']

Using device: cuda
GPU: NVIDIA RTX A6000


In [4]:
# Cell 2 — Shared scaffolding: regression tensors
df_reg = mat.dropna(subset=[TARGET_REG])
tr_reg, vl_reg, te_reg = chronological_split(df_reg)

X_train_r = tr_reg[features].ffill().fillna(0);  y_train_r = tr_reg[TARGET_REG]
X_val_r   = vl_reg[features].ffill().fillna(0);  y_val_r   = vl_reg[TARGET_REG]
X_test_r  = te_reg[features].ffill().fillna(0);  y_test_r  = te_reg[TARGET_REG]

print("Regression shapes:")
print(f"  X_train_r={X_train_r.shape}  y_train_r={y_train_r.shape}")
print(f"  X_val_r=  {X_val_r.shape}  y_val_r=  {y_val_r.shape}")
print(f"  X_test_r= {X_test_r.shape}  y_test_r= {y_test_r.shape}")

  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,428 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Regression shapes:
  X_train_r=(60428, 63)  y_train_r=(60428,)
  X_val_r=  (8760, 63)  y_val_r=  (8760,)
  X_test_r= (8767, 63)  y_test_r= (8767,)


In [5]:
# Cell 3 — Shared scaffolding: classification tensors
df_clf = mat.dropna(subset=[TARGET_CLASS])
tr_clf, vl_clf, te_clf = chronological_split(df_clf)

X_train_c = tr_clf[features].ffill().fillna(0);  y_train_c = tr_clf[TARGET_CLASS]
X_val_c   = vl_clf[features].ffill().fillna(0);  y_val_c   = vl_clf[TARGET_CLASS]
X_test_c  = te_clf[features].ffill().fillna(0);  y_test_c  = te_clf[TARGET_CLASS]

print("Classification shapes:")
print(f"  X_train_c={X_train_c.shape}  y_train_c={y_train_c.shape}")
print(f"  X_val_c=  {X_val_c.shape}  y_val_c=  {y_val_c.shape}")
print(f"  X_test_c= {X_test_c.shape}  y_test_c= {y_test_c.shape}")

  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Classification shapes:
  X_train_c=(60429, 63)  y_train_c=(60429,)
  X_val_c=  (8760, 63)  y_val_c=  (8760,)
  X_test_c= (8767, 63)  y_test_c= (8767,)


In [6]:
# Cell A — Windowing helper
def make_windows(X, y, lookback=168):
    """Return previous-`lookback` feature rows and the next-hour target.

    Windowing is applied independently inside train/val/test after the
    chronological split, so no sequence crosses a split boundary.
    """
    X_values = np.asarray(X, dtype=np.float32)
    y_values = np.asarray(y, dtype=np.float32)
    if len(X_values) != len(y_values):
        raise ValueError(f"X/y length mismatch: {len(X_values)} != {len(y_values)}")
    if len(X_values) <= lookback:
        raise ValueError(f"Need more than lookback={lookback} rows, got {len(X_values)}")

    n = len(X_values) - lookback
    X_seq = np.empty((n, lookback, X_values.shape[1]), dtype=np.float32)
    y_seq = np.empty(n, dtype=np.float32)
    for out_i, target_i in enumerate(range(lookback, len(X_values))):
        X_seq[out_i] = X_values[target_i - lookback:target_i]
        y_seq[out_i] = y_values[target_i]
    return X_seq, y_seq

# Tiny shape smoke test: previous 3 rows -> next target.
_X_demo = pd.DataFrame(np.arange(20, dtype=np.float32).reshape(10, 2))
_y_demo = pd.Series(np.arange(10, dtype=np.float32))
_Xw_demo, _yw_demo = make_windows(_X_demo, _y_demo, lookback=3)
assert _Xw_demo.shape == (7, 3, 2)
assert _yw_demo.shape == (7,)
assert np.all(_Xw_demo[0] == _X_demo.values[:3])
assert _yw_demo[0] == _y_demo.iloc[3]
print("make_windows smoke test passed:", _Xw_demo.shape, _yw_demo.shape)

make_windows smoke test passed: (7, 3, 2) (7,)


In [7]:
# Cell B — StandardScaler fit on train only; transform windows without leakage
def scale_window_features(X_train_seq, X_val_seq, X_test_seq):
    n_features = X_train_seq.shape[-1]
    x_scaler = StandardScaler()
    x_scaler.fit(X_train_seq.reshape(-1, n_features))

    def transform(X_seq):
        original_shape = X_seq.shape
        scaled = x_scaler.transform(X_seq.reshape(-1, n_features))
        return scaled.reshape(original_shape).astype(np.float32)

    return transform(X_train_seq), transform(X_val_seq), transform(X_test_seq), x_scaler

def prepare_regression_sequences(lookback):
    X_train_seq, y_train_seq = make_windows(X_train_r, y_train_r, lookback=lookback)
    X_val_seq, y_val_seq = make_windows(X_val_r, y_val_r, lookback=lookback)
    X_test_seq, y_test_seq = make_windows(X_test_r, y_test_r, lookback=lookback)

    X_train_seq, X_val_seq, X_test_seq, x_scaler = scale_window_features(
        X_train_seq, X_val_seq, X_test_seq
    )

    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train_seq.reshape(-1, 1)).ravel().astype(np.float32)
    y_val_scaled = y_scaler.transform(y_val_seq.reshape(-1, 1)).ravel().astype(np.float32)
    y_test_scaled = y_scaler.transform(y_test_seq.reshape(-1, 1)).ravel().astype(np.float32)

    scalers = {"x": x_scaler, "y": y_scaler}
    return {
        "X_train": X_train_seq, "y_train": y_train_scaled, "y_train_raw": y_train_seq,
        "X_val": X_val_seq, "y_val": y_val_scaled, "y_val_raw": y_val_seq,
        "X_test": X_test_seq, "y_test": y_test_scaled, "y_test_raw": y_test_seq,
        "scalers": scalers,
    }

def prepare_classification_sequences(lookback):
    X_train_seq, y_train_seq = make_windows(X_train_c, y_train_c, lookback=lookback)
    X_val_seq, y_val_seq = make_windows(X_val_c, y_val_c, lookback=lookback)
    X_test_seq, y_test_seq = make_windows(X_test_c, y_test_c, lookback=lookback)

    X_train_seq, X_val_seq, X_test_seq, x_scaler = scale_window_features(
        X_train_seq, X_val_seq, X_test_seq
    )

    return {
        "X_train": X_train_seq, "y_train": y_train_seq.astype(np.float32),
        "X_val": X_val_seq, "y_val": y_val_seq.astype(np.float32),
        "X_test": X_test_seq, "y_test": y_test_seq.astype(np.float32),
        "scalers": {"x": x_scaler},
    }

# Default target config. If CUDA OOM occurs during training below, the next cell
# automatically retries with lookback=72; if needed, it also drops hidden_size to 32.
LOOKBACK = 168
HIDDEN_SIZE = 64

_reg_demo = prepare_regression_sequences(lookback=LOOKBACK)
print("Regression sequence shapes:")
for key in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]:
    print(f"  {key}: {_reg_demo[key].shape}")
regression_scalers = _reg_demo["scalers"]
del _reg_demo
gc.collect()

Regression sequence shapes:
  X_train: (60260, 168, 63)
  y_train: (60260,)
  X_val: (8592, 168, 63)
  y_val: (8592,)
  X_test: (8599, 168, 63)
  y_test: (8599,)


25

In [8]:
# Cell C — Bidirectional LSTM module
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

# Same backbone/head shape works for regression logits and binary-classifier logits.
_input_size = len(features)
_smoke_model = LSTMForecaster(input_size=_input_size, hidden_size=64)
_smoke_x = torch.zeros(2, 168, _input_size)
_smoke_y = _smoke_model(_smoke_x)
assert _smoke_y.shape == (2,)
print(_smoke_model)
print("Forward smoke test passed:", tuple(_smoke_y.shape))
del _smoke_model, _smoke_x, _smoke_y

LSTMForecaster(
  (lstm): LSTM(63, 64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (head): Linear(in_features=128, out_features=1, bias=True)
)
Forward smoke test passed: (2,)


In [9]:
# Cell D — Training loop: AdamW, early stopping on validation MAE
def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).float())
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(device == "cuda"),
    )

@torch.no_grad()
def predict_numpy(model, X, batch_size=512):
    model.eval()
    preds = []
    loader = make_loader(X, np.zeros(len(X), dtype=np.float32), batch_size=batch_size, shuffle=False)
    for xb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)

def train_regressor_for_config(lookback, hidden_size, max_epochs=30, patience=5, batch_size=256):
    seq = prepare_regression_sequences(lookback=lookback)
    model = LSTMForecaster(
        input_size=seq["X_train"].shape[-1],
        hidden_size=hidden_size,
        num_layers=2,
        dropout=0.2,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()
    train_loader = make_loader(seq["X_train"], seq["y_train"], batch_size=batch_size, shuffle=True)

    best_state = None
    best_val_mae = float("inf")
    stale_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_pred_scaled = predict_numpy(model, seq["X_val"])
        val_pred = seq["scalers"]["y"].inverse_transform(val_pred_scaled.reshape(-1, 1)).ravel()
        val_mae = np.mean(np.abs(seq["y_val_raw"] - val_pred))
        train_loss = float(np.mean(train_losses))
        print(f"epoch {epoch:02d} | train MSE(scaled): {train_loss:.4f} | val MAE: ${val_mae:.2f}/MWh")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping after {epoch} epochs; best val MAE=${best_val_mae:.2f}/MWh")
                break

    model.load_state_dict(best_state)
    return model, seq, best_val_mae

REGRESSION_CONFIGS = [
    (168, 64),
    (72, 64),   # OOM fallback: shorter sequence.
    (72, 32),   # OOM fallback: smaller LSTM.
]

lstm_reg = None
reg_seq = None
LOOKBACK_USED = None
HIDDEN_SIZE_USED = None
OOM_MITIGATION = "none"

for lookback, hidden_size in REGRESSION_CONFIGS:
    try:
        print(f"\nTraining regression LSTM with lookback={lookback}, hidden_size={hidden_size}")
        lstm_reg, reg_seq, best_val_mae = train_regressor_for_config(lookback, hidden_size)
        LOOKBACK_USED = lookback
        HIDDEN_SIZE_USED = hidden_size
        if (lookback, hidden_size) == (72, 64):
            OOM_MITIGATION = "dropped lookback from 168h to 72h"
        elif (lookback, hidden_size) == (72, 32):
            OOM_MITIGATION = "dropped lookback from 168h to 72h and hidden_size from 64 to 32"
        break
    except RuntimeError as exc:
        is_oom = "out of memory" in str(exc).lower()
        if device == "cuda" and is_oom:
            print(f"CUDA OOM at lookback={lookback}, hidden_size={hidden_size}; trying fallback config.")
            del lstm_reg
            lstm_reg = None
            reg_seq = None
            gc.collect()
            torch.cuda.empty_cache()
            continue
        raise

if lstm_reg is None:
    raise RuntimeError("All LSTM regression configs failed.")

regression_scalers = reg_seq["scalers"]
print(f"\nSelected regression config: lookback={LOOKBACK_USED}, hidden_size={HIDDEN_SIZE_USED}")
print(f"OOM mitigation applied: {OOM_MITIGATION}")


Training regression LSTM with lookback=168, hidden_size=64
epoch 01 | train MSE(scaled): 0.4500 | val MAE: $66.57/MWh
epoch 02 | train MSE(scaled): 0.1713 | val MAE: $65.54/MWh
epoch 03 | train MSE(scaled): 0.1355 | val MAE: $66.97/MWh
epoch 04 | train MSE(scaled): 0.1286 | val MAE: $62.09/MWh
epoch 05 | train MSE(scaled): 0.1135 | val MAE: $61.97/MWh
epoch 06 | train MSE(scaled): 0.1156 | val MAE: $55.45/MWh
epoch 07 | train MSE(scaled): 0.0966 | val MAE: $56.90/MWh
epoch 08 | train MSE(scaled): 0.1037 | val MAE: $61.66/MWh
epoch 09 | train MSE(scaled): 0.0919 | val MAE: $56.92/MWh
epoch 10 | train MSE(scaled): 0.0912 | val MAE: $55.74/MWh
epoch 11 | train MSE(scaled): 0.0776 | val MAE: $57.92/MWh
Early stopping after 11 epochs; best val MAE=$55.45/MWh

Selected regression config: lookback=168, hidden_size=64
OOM mitigation applied: none


In [10]:
# Cell E — Regression evaluation on test set
y_pred_test_scaled = predict_numpy(lstm_reg, reg_seq["X_test"])
y_pred_test_r_lstm = regression_scalers["y"].inverse_transform(y_pred_test_scaled.reshape(-1, 1)).ravel()
y_test_r_lstm = reg_seq["y_test_raw"]

print("\nTest-set metrics:")
regression_report(y_test_r_lstm, y_pred_test_r_lstm, name="lstm_v2_regressor")

print(f"lookback used: {LOOKBACK_USED}h")
print(f"hidden size used: {HIDDEN_SIZE_USED}")
print(f"OOM mitigation applied: {OOM_MITIGATION}")


Test-set metrics:

--- Regression report: lstm_v2_regressor ---
  MAE:             $32.45/MWh
  RMSE:            $134.37/MWh
  Spike recall:    15.79%
  Spike precision: 10.84%
---------------------------------

lookback used: 168h
hidden size used: 64
OOM mitigation applied: none


In [11]:
# Cell F — Optional binary classifier using the same LSTM backbone
def train_classifier_for_config(lookback, hidden_size, max_epochs=30, patience=5, batch_size=256):
    seq = prepare_classification_sequences(lookback=lookback)
    model = LSTMForecaster(
        input_size=seq["X_train"].shape[-1],
        hidden_size=hidden_size,
        num_layers=2,
        dropout=0.2,
    ).to(device)

    pos_rate = float(seq["y_train"].mean())
    pos_weight_value = (1.0 - pos_rate) / max(pos_rate, 1e-8)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    print(f"Positive class rate after windowing: {pos_rate:.4f}")
    print(f"BCE pos_weight: {pos_weight_value:.2f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = make_loader(seq["X_train"], seq["y_train"], batch_size=batch_size, shuffle=True)

    best_state = None
    best_val_loss = float("inf")
    stale_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_logits = predict_numpy(model, seq["X_val"])
        val_targets = torch.from_numpy(seq["y_val"]).float().to(device)
        val_logits_t = torch.from_numpy(val_logits).float().to(device)
        val_loss = float(loss_fn(val_logits_t, val_targets).detach().cpu().item())
        print(f"epoch {epoch:02d} | train BCE: {np.mean(train_losses):.4f} | val BCE: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping after {epoch} epochs; best val BCE={best_val_loss:.4f}")
                break

    model.load_state_dict(best_state)
    return model, seq, best_val_loss

try:
    print(f"Training classifier LSTM with lookback={LOOKBACK_USED}, hidden_size={HIDDEN_SIZE_USED}")
    lstm_clf, clf_seq, best_val_bce = train_classifier_for_config(LOOKBACK_USED, HIDDEN_SIZE_USED)
except RuntimeError as exc:
    is_oom = "out of memory" in str(exc).lower()
    if device == "cuda" and is_oom and (LOOKBACK_USED, HIDDEN_SIZE_USED) != (72, 32):
        print("CUDA OOM in classifier; retrying with lookback=72, hidden_size=32.")
        torch.cuda.empty_cache()
        gc.collect()
        lstm_clf, clf_seq, best_val_bce = train_classifier_for_config(72, 32)
    else:
        raise

test_logits = predict_numpy(lstm_clf, clf_seq["X_test"])
test_proba_lstm = 1.0 / (1.0 + np.exp(-test_logits))

print("\nTest-set metrics:")
classification_report(clf_seq["y_test"], test_proba_lstm, name="lstm_v2_classifier", threshold=0.5)

Training classifier LSTM with lookback=168, hidden_size=64
Positive class rate after windowing: 0.2460
BCE pos_weight: 3.06
epoch 01 | train BCE: 0.8856 | val BCE: 1.9634
epoch 02 | train BCE: 0.5464 | val BCE: 3.0149
epoch 03 | train BCE: 0.3717 | val BCE: 3.4026
epoch 04 | train BCE: 0.2749 | val BCE: 4.0157
epoch 05 | train BCE: 0.2231 | val BCE: 3.9060
epoch 06 | train BCE: 0.1823 | val BCE: 4.3548
Early stopping after 6 epochs; best val BCE=1.9634

Test-set metrics:

--- Classification report: lstm_v2_classifier ---
  PR-AUC:    0.213
  ROC-AUC:   0.601
  Recall:    36.05%  (at threshold 0.5)
  Precision: 22.60%
  Confusion: TP=530  FP=1815  FN=940  TN=5314
-------------------------------------



{'name': 'lstm_v2_classifier',
 'pr_auc': 0.21335515569505148,
 'roc_auc': 0.6014573987822089,
 'recall': np.float64(0.36054421768707484),
 'precision': np.float64(0.2260127931769723),
 'tp': 530,
 'fp': 1815,
 'fn': 940,
 'tn': 5314}

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np
_y_true_dollar = np.asarray(y_test_r_lstm).reshape(-1)
_y_pred_dollar = np.asarray(y_pred_test_r_lstm).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC: {_spike_pr_auc:.3f}")